# 03 - Placeholders and Background

Validate placeholder-first workflow and slide background mutation tools.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python").exists() and (REPO_ROOT.parent / "python").exists():
    REPO_ROOT = REPO_ROOT.parent
PYTHON_ROOT = REPO_ROOT / "python"
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))


def build_helpers():
    from errors import BridgeError
    from service import PowerPointService

    return BridgeError, PowerPointService


BridgeError, PowerPointService = build_helpers()
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebook-tests"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
service = PowerPointService()
pprint(service.dispatch("pptx_get_engine_info", {}))

In [ ]:
create_result = service.dispatch("pptx_create_presentation", {"width": "10in", "height": "5.625in"})
presentation_id = create_result["presentation_id"]

layouts = service.dispatch("pptx_get_layouts", {"presentation_id": presentation_id})["layouts"]
layout = next(
    (layout_candidate for layout_candidate in layouts if layout_candidate.get("placeholder_types")), layouts[0]
)
layout_name = layout["name"]

add = service.dispatch("pptx_add_slide", {"presentation_id": presentation_id, "layout_name": layout_name})
slide_index = int(add["added_slide_index"])
placeholders = service.dispatch(
    "pptx_get_placeholders", {"presentation_id": presentation_id, "slide_index": slide_index}
)["placeholders"]
print("Layout used:", layout_name)
print("Placeholder count:", len(placeholders))

In [ ]:
selected_placeholder = None
for ph in placeholders:
    candidate = ph["name"]
    try:
        service.dispatch(
            "pptx_set_placeholder_text",
            {
                "presentation_id": presentation_id,
                "slide_index": slide_index,
                "placeholder_name": candidate,
                "text_content": "Notebook placeholder text check",
                "alignment": "left",
                "bold": True,
            },
        )
        selected_placeholder = candidate
        break
    except BridgeError as exc:
        if exc.code in {"conflict", "validation_error", "not_found"}:
            continue
        raise

if selected_placeholder is None:
    raise RuntimeError("No text-capable placeholder found on this layout")

text_result = service.dispatch(
    "pptx_get_placeholder_text",
    {
        "presentation_id": presentation_id,
        "slide_index": slide_index,
        "placeholder_name": selected_placeholder,
    },
)
print("Selected placeholder:", selected_placeholder)
print("Raw text:", text_result.get("raw_text"))

service.dispatch(
    "pptx_clear_placeholder",
    {
        "presentation_id": presentation_id,
        "slide_index": slide_index,
        "placeholder_name": selected_placeholder,
    },
)
print("Placeholder cleared")

In [ ]:
service.dispatch(
    "pptx_set_slide_background",
    {
        "presentation_id": presentation_id,
        "slide_index": slide_index,
        "color_hex": "#F5F9FF",
    },
)

service.dispatch(
    "pptx_set_slide_background",
    {
        "presentation_id": presentation_id,
        "slide_index": slide_index,
        "gradient_start_color_hex": "#FFFFFF",
        "gradient_end_color_hex": "#DCEEFF",
        "gradient_angle_deg": 90,
    },
)

print("Background mutation calls completed")

Optional image placeholder test:

Set `IMAGE_PATH` to an absolute image path on your machine.

In [ ]:
IMAGE_PATH = ""  # Example: r'C:\\Users\\you\\Pictures\\sample.jpg'

if IMAGE_PATH:
    # Reuse selected placeholder only if it supports image insertion in your template.
    service.dispatch(
        "pptx_set_placeholder_image",
        {
            "presentation_id": presentation_id,
            "slide_index": slide_index,
            "placeholder_name": selected_placeholder,
            "image_path": str(Path(IMAGE_PATH).resolve()),
        },
    )
    print("Image placeholder set from:", IMAGE_PATH)
else:
    print("Skipped image placeholder test (IMAGE_PATH is empty).")

In [ ]:
output_path = (ARTIFACT_DIR / "nb03_placeholders_background_output.pptx").resolve()
service.dispatch("pptx_save_presentation", {"presentation_id": presentation_id, "output_path": str(output_path)})
service.dispatch("pptx_close_presentation", {"presentation_id": presentation_id})
service.shutdown()
print("Saved:", output_path)